# Generate_Descriptors
Compute per-ligand descriptors for the CN/NN project.
- Reads `ML-CN-Smiles.csv` and `ML-NN-Smiles.csv`.
- Loads LoQI SDF conformer files from `output/CN_conformers` and `output/NN_conformers`.
- Writes temporary XYZs for downstream 3D tools.
- Computes basic RDKit 2D descriptors.
- Provides hooks for `morfeus` and `xtb` descriptor functions (no external packages required to open/run this notebook).

## User Options

In [ ]:
from pathlib import Path

BASE_DIR = Path(".").resolve()
IN_DIR = BASE_DIR / "input"
OUT_DIR = BASE_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Input files
CN_CSV =        IN_DIR / "ML-CN-Smiles.csv"
NN_CSV =        IN_DIR / "ML-NN-Smiles.csv"
TARGET_CSV =    IN_DIR / "inflection_points_activity_score_log.csv"

# Output files
OUT_LIGAND_FILE =       OUT_DIR / "CN_NN_ligand_descriptors.csv"
OUT_DESCRIPTORS_FILE =  OUT_DIR / "CN_NN_combinations_descriptors.csv"
OUTPUT_CSV =            OUT_DIR / "CN_NN_combinations_descriptors_with_score.csv"

## Imports and Configuration

In [ ]:
import json
import re
import pandas as pd
import numpy as np

# RDKit availability and common symbols
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, Lipinski
    RDKIT_AVAILABLE = True
except Exception:
    RDKIT_AVAILABLE = False

# Morfeus / xTB availability
try:
    from morfeus import read_xyz
    MORFEUS_AVAILABLE = True
except Exception:
    MORFEUS_AVAILABLE = False

try:
    from morfeus import XTB  # optional
    XTB_AVAILABLE = True
except Exception:
    XTB_AVAILABLE = False

## Mordred
try:
    from mordred import Calculator
    from mordred import (
        EState,
        BCUT,
        TopologicalCharge,
        Polarizability,
        Autocorrelation,
        Chi,
        BalabanJ,
        WienerIndex,
        ZagrebIndex,
        KappaShapeIndex,
        McGowanVolume,
        VdwVolumeABC,
    )
    from mordred.error import Error, Missing
    MORDRED_AVAILABLE = True
except Exception:
    MORDRED_AVAILABLE = False

print('RDKIT_AVAILABLE =', RDKIT_AVAILABLE)
print('MORFEUS_AVAILABLE =', MORFEUS_AVAILABLE)
print('XTB_AVAILABLE =', XTB_AVAILABLE)
print('MORDRED_AVAILABLE =', MORDRED_AVAILABLE)


RDKIT_AVAILABLE = True
MORFEUS_AVAILABLE = True
XTB_AVAILABLE = True


In [ ]:
def read_sdf_pick_lowest_energy(sdf_path):
    """Return (rdkit_mol_with_conformer, energy) picking lowest-energy conformer if energy present."""
    supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=False)
    mols = [m for m in supplier if m is not None]
    if not mols:
        return None, None
    mol_energy_pairs = []
    for m in mols:
        energy = None
        try:
            props = m.GetPropsAsDict()
            for v in props.values():
                # try numeric property first
                try:
                    val = float(v)
                    energy = val if (energy is None or val < energy) else energy
                except Exception:
                    s = str(v)
                    mm = re.search(r'(-?\d+\.\d+)', s)
                    if mm:
                        try:
                            val = float(mm.group(1))
                            energy = val if (energy is None or val < energy) else energy
                        except Exception:
                            pass
        except Exception:
            # some SD blocks may not have properties
            pass
        mol_energy_pairs.append((m, energy))
    energies = [(m,e) for m,e in mol_energy_pairs if e is not None]
    if energies:
        mol_sel, e_sel = min(energies, key=lambda x: x[1])
    else:
        mol_sel, e_sel = mol_energy_pairs[0]
    return mol_sel, e_sel


def write_xyz(mol, out_path, energy=None):
    conf = mol.GetConformer()
    n = mol.GetNumAtoms()
    lines = [str(n), f"Energy {energy}" if energy is not None else "Converted from SDF"]
    for i in range(n):
        a = mol.GetAtomWithIdx(i)
        p = conf.GetAtomPosition(i)
        lines.append(f"{a.GetSymbol()} {p.x:.6f} {p.y:.6f} {p.z:.6f}")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(lines))
    return str(out_path)

print('Helper functions defined: read_sdf_pick_lowest_energy, write_xyz')

Helper functions defined: read_sdf_pick_lowest_energy, write_xyz


## RDKit
| Descriptor column   | Package/source                             |
| ------------------- | ------------------------------------------ |
| `MolWt`             | RDKit `Descriptors.MolWt`                  |
| `LogP`              | RDKit `Crippen.MolLogP`                    |
| `TPSA`              | RDKit `rdMolDescriptors.CalcTPSA`          |
| `NumHDonors`        | RDKit `Lipinski.NumHDonors`                |
| `NumHAcceptors`     | RDKit `Lipinski.NumHAcceptors`             |
| `NumRotatableBonds` | RDKit `Lipinski.NumRotatableBonds`         |
| `NumHeavyAtoms`     | RDKit `rdMolDescriptors.CalcNumHeavyAtoms` |


In [ ]:
# RDKit descriptor functions
def compute_rdkit_descriptors(mol):
    if mol is None:
        return {}
    try:
        desc = {}
        desc['MolWt'] = float(Descriptors.MolWt(mol))
        desc['LogP'] = float(Crippen.MolLogP(mol))
        desc['TPSA'] = float(rdMolDescriptors.CalcTPSA(mol))
        desc['NumHDonors'] = int(Lipinski.NumHDonors(mol))
        desc['NumHAcceptors'] = int(Lipinski.NumHAcceptors(mol))
        desc['NumRotatableBonds'] = int(Lipinski.NumRotatableBonds(mol))
        desc['NumHeavyAtoms'] = int(rdMolDescriptors.CalcNumHeavyAtoms(mol))
        return desc
    except Exception as e:
        return {'rdkit_error': str(e)}


print('RDKit descriptor functions ready')

RDKit descriptor functions ready


## Mordred
| Descriptor column             | Package/source                      |
| ----------------------------- | ----------------------------------- |
| `Mordred_EState_*`            | Mordred `mordred.EState`            |
| `Mordred_BCUT_*`              | Mordred `mordred.BCUT`              |
| `Mordred_TopologicalCharge_*` | Mordred `mordred.TopologicalCharge` |
| `Mordred_Polarizability_*`    | Mordred `mordred.Polarizability`    |
| `Mordred_Autocorrelation_*`   | Mordred `mordred.Autocorrelation`   |
| `Mordred_Chi_*`               | Mordred `mordred.Chi`               |
| `Mordred_BalabanJ_*`          | Mordred `mordred.BalabanJ`          |
| `Mordred_WienerIndex_*`       | Mordred `mordred.WienerIndex`       |
| `Mordred_ZagrebIndex_*`       | Mordred `mordred.ZagrebIndex`       |
| `Mordred_KappaShapeIndex_*`   | Mordred `mordred.KappaShapeIndex`   |
| `Mordred_McGowanVolume`       | Mordred `mordred.McGowanVolume`     |
| `Mordred_VdwVolumeABC`        | Mordred `mordred.VdwVolumeABC`      |

In [ ]:
# Mordred descriptor functions (selected)
def compute_mordred_descriptors(mol):
    if mol is None:
        return {}
    
    try:
        desc = {}

        selected_mordred_modules = [
            EState,
            BCUT,
            TopologicalCharge,
            Polarizability,
            Autocorrelation,
            Chi,
            BalabanJ,
            WienerIndex,
            ZagrebIndex,
            KappaShapeIndex,
            McGowanVolume,
            VdwVolumeABC,
        ]

        calc = Calculator(selected_mordred_modules, ignore_3D=True)
        result = calc(mol)

        for key, value in result.asdict().items():
            col = f"Mordred_{key}"

            # Skip Mordred calculation errors/missing values
            if isinstance(value, (Error, Missing)):
                continue

            # Keep only numeric values
            try:
                value = float(value)
                if np.isfinite(value):
                    desc[col] = value
            except Exception:
                continue

        return desc

    except Exception as e:
        return {"mordred_error": str(e)}

print("Mordred descriptor functions ready")

## Morfeus Descriptors
| Descriptor column                 | Package/source                                                            |
| --------------------------------- | ------------------------------------------------------------------------- |
| `Morfeus_Pint`                    | Morfeus `Dispersion`, from `disp.p_int`                                   |
| `Morfeus_DispersionArea`          | Morfeus `Dispersion`, from `disp.area`                                    |
| `Morfeus_DispersionVol`           | Morfeus `Dispersion`, from `disp.volume`                                  |
| `Morfeus_SASA`                    | Morfeus `SASA`, from `sasa_obj.area`                                      |
| `Morfeus_SASAVolume`              | Morfeus `SASA`, from `sasa_obj.volume`                                    |
| `Morfeus_SASA_atom_areas`         | Morfeus `SASA`, from `sasa_obj.atom_areas`                                |
| `Morfeus_BuriedVolume_Calpha`     | Morfeus `BuriedVolume`; CN ligand reaction center                         |
| `Morfeus_BuriedVolume_N`          | Morfeus `BuriedVolume`; NN ligand reaction center                         |
| `Morfeus_Sterimol_L`              | Morfeus `Sterimol`, from `s.L_value`                                      |
| `Morfeus_Sterimol_B1`             | Morfeus `Sterimol`, from `s.B_1_value`                                    |
| `Morfeus_Sterimol_B5`             | Morfeus `Sterimol`, from `s.B_5_value`                                    |
| `Morfeus_SolidAngle_steradians`   | Morfeus `SolidAngle`, from `sa.solid_angle`                               |
| `Morfeus_SolidAngle_cone_deg`     | Morfeus `SolidAngle`, from `sa.cone_angle`                                |
| `Morfeus_SolidAngle_G`            | Morfeus `SolidAngle`, from `sa.G`                                         |
| `Morfeus_ConeAngle_deg`           | Morfeus `ConeAngle`, from `ca.cone_angle`                                 |
| `Morfeus_ConeAngle_tangent_atoms` | Morfeus `ConeAngle`, from `ca.tangent_atoms`                              |
| `Morfeus_BiteAngle_deg`           | Morfeus `BiteAngle`, from `ba.angle`                                      |
| `Morfeus_BiteAngle_inverted`      | Morfeus `BiteAngle`, from `ba.inverted`                                   |
| `Morfeus_BiteAngle_deg_approx`    | fallback NumPy geometric approximation, only if Morfeus `BiteAngle` fails |


In [ ]:
def _read_morfeus_geometry(xyz_file):
    elements, coords = read_xyz(str(xyz_file))
    if hasattr(coords, 'ndim') and coords.ndim == 3:
        coords = coords[0]
    return list(elements), np.asarray(coords)


def find_reaction_center(elements, coordinates, mol_type):
    """
    Find 1-indexed reaction center and dummy atom for CN/NN ligands.
    Returns dict with keys 'center_idx' and 'dummy_idx' (1-indexed) or {}.
    """
    coords = np.asarray(coordinates)
    if mol_type == 'CN':
        br_indices = [i + 1 for i, e in enumerate(elements) if str(e).capitalize() == 'Br']
        if not br_indices:
            return {}
        c_indices = [i + 1 for i, e in enumerate(elements) if str(e).capitalize() == 'C']
        if not c_indices:
            return {}
        def alpha_c_for_br(br_1idx):
            br_coord = coords[br_1idx - 1]
            return min(c_indices, key=lambda ci: np.linalg.norm(coords[ci - 1] - br_coord))
        def is_sp3_carbon(c_1idx):
            c_coord = coords[c_1idx - 1]
            aromatic_cc = sum(
                1
                for i, e in enumerate(elements)
                if str(e).capitalize() == 'C' and i != (c_1idx - 1)
                and 1.30 < np.linalg.norm(coords[i] - c_coord) < 1.45
            )
            return aromatic_cc < 2
        chosen_br = br_indices[0]
        chosen_c = alpha_c_for_br(chosen_br)
        for br_1idx in br_indices:
            c_alpha = alpha_c_for_br(br_1idx)
            if is_sp3_carbon(c_alpha):
                chosen_br = br_1idx
                chosen_c = c_alpha
                break
        return {'center_idx': chosen_c, 'dummy_idx': chosen_br}

    elif mol_type == 'NN':
        n_indices = [i + 1 for i, e in enumerate(elements) if str(e).capitalize() == 'N']
        if not n_indices:
            return {}
        h_indices = [i + 1 for i, e in enumerate(elements) if str(e).capitalize() == 'H']
        def h_count_on_n(n_1idx):
            n_coord = coords[n_1idx - 1]
            return sum(1 for hi in h_indices if np.linalg.norm(coords[hi - 1] - n_coord) < 1.3)
        def n_priority(n_1idx):
            nh = h_count_on_n(n_1idx)
            if nh == 1:
                return 0
            elif nh >= 2:
                return 1
            else:
                return 2
        best_n_idx = min(n_indices, key=n_priority)
        n_coord = coords[best_n_idx - 1]
        h_on_n_idx = None
        if h_indices:
            closest_h = min(h_indices, key=lambda hi: np.linalg.norm(coords[hi - 1] - n_coord))
            if np.linalg.norm(coords[closest_h - 1] - n_coord) < 1.3:
                h_on_n_idx = closest_h
        return {'center_idx': best_n_idx, 'dummy_idx': h_on_n_idx}

    return {}


# Ligand-only Morfeus descriptors: Dispersion (P_int), SASA, Buried Volume, Sterimol

def morfeus_dispersion(xyz_file):
    try:
        from morfeus import Dispersion
    except Exception:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    try:
        disp = Dispersion(elements, coords)
        return {
            'Morfeus_Pint': float(disp.p_int),
            'Morfeus_DispersionArea': float(disp.area),
            'Morfeus_DispersionVol': float(disp.volume),
        }
    except Exception:
        return {}


def morfeus_sasa(xyz_file):
    try:
        from morfeus import SASA
    except Exception:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    try:
        sasa_obj = SASA(elements, coords)
        out = {'Morfeus_SASA': float(sasa_obj.area), 'Morfeus_SASAVolume': float(sasa_obj.volume)}
        # per-atom areas available as sasa_obj.atom_areas (1-indexed keys)
        try:
            out['Morfeus_SASA_atom_areas'] = dict(sasa_obj.atom_areas)
        except Exception:
            pass
        return out
    except Exception:
        return {}


def morfeus_buried_volume(xyz_file, center_idx=None, mol_type=None):
    try:
        from morfeus import BuriedVolume
    except Exception:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    if center_idx is None and mol_type is not None:
        centers = find_reaction_center(elements, coords, mol_type)
        center_idx = centers.get('center_idx')
    if not center_idx:
        return {}
    try:
        bv = BuriedVolume(elements, coords, center_idx)
        label = 'Calpha' if mol_type == 'CN' else 'N'
        return {f'Morfeus_BuriedVolume_{label}': float(bv.fraction_buried_volume * 100)}
    except Exception:
        return {}


def morfeus_sterimol(xyz_file, center_idx=None, dummy_idx=None, mol_type=None):
    try:
        from morfeus import Sterimol
    except Exception:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    if (center_idx is None or dummy_idx is None) and mol_type is not None:
        centers = find_reaction_center(elements, coords, mol_type)
        center_idx = center_idx or centers.get('center_idx')
        dummy_idx = dummy_idx or centers.get('dummy_idx')
    if not center_idx or not dummy_idx:
        return {}
    try:
        s = Sterimol(elements, coords, dummy_idx, center_idx)
        return {'Morfeus_Sterimol_L': float(s.L_value), 'Morfeus_Sterimol_B1': float(s.B_1_value), 'Morfeus_Sterimol_B5': float(s.B_5_value)}
    except Exception:
        return {}

# Complex-specific Morfeus descriptors: Solid/Cone Angle, Bite Angle
# Keep these functions for now, but not called for ligand-only XYZ files.

def morfeus_solid_and_cone(xyz_file, metal_index=None, mol_type=None):
    """
    Compute solid angle and cone angle.

    Only use this for full Ir complexes or structures with a real metal center. Not for ligand-only XYZ files.
    """
    try:
        from morfeus import SolidAngle, ConeAngle
    except Exception:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    if metal_index is None: return {}
    out = {}
    try:
        sa = SolidAngle(elements, coords, metal_index)
        out['Morfeus_SolidAngle_steradians'] = float(sa.solid_angle)
        out['Morfeus_SolidAngle_cone_deg'] = float(sa.cone_angle)
        out['Morfeus_SolidAngle_G'] = float(sa.G)
    except Exception:
        pass
    try:
        ca = ConeAngle(elements, coords, metal_index)
        out['Morfeus_ConeAngle_deg'] = float(ca.cone_angle)
        try:
            out['Morfeus_ConeAngle_tangent_atoms'] = list(ca.tangent_atoms)
        except Exception:
            pass
    except Exception:
        pass
    return out


def morfeus_bite_angle(xyz_file, metal_index=None, ligand_idx1=None, ligand_idx2=None, ref_atoms=None, mol_type=None):
    try:
        from morfeus import BiteAngle
    except Exception:
        return {}
    elements, coords = _read_morfeus_geometry(xyz_file)
    # guess metal
    if metal_index is None: return {}
    # guess donor atoms if not supplied
    if ligand_idx1 is None or ligand_idx2 is None:
        donors = [i + 1 for i, e in enumerate(elements) if str(e).capitalize() == 'N']
        if len(donors) >= 2:
            ligand_idx1, ligand_idx2 = donors[0], donors[1]
        else:
            # cannot determine bite angle
            return {}
    try:
        ba = BiteAngle(coords, metal_index, ligand_idx1, ligand_idx2, ref_atoms=ref_atoms)
        return {'Morfeus_BiteAngle_deg': float(ba.angle), 'Morfeus_BiteAngle_inverted': bool(ba.inverted)}
    except Exception:
        return {}

Appended Morfeus/xTB advanced feature functions


| Descriptor column        | Package/source                                          |
| ------------------------ | ------------------------------------------------------- |
| `xTB_HOMO`               | Morfeus `XTB.get_homo()`                                |
| `xTB_LUMO`               | Morfeus `XTB.get_lumo()`                                |
| `xTB_HOMO_LUMO_gap`      | computed from `LUMO - HOMO`                             |
| `xTB_IP`                 | Morfeus `XTB.get_ip(corrected=True)`                    |
| `xTB_EA`                 | Morfeus `XTB.get_ea(corrected=True)`                    |
| `xTB_Electrophilicity`   | Morfeus `XTB.get_global_descriptor("electrophilicity")` |
| `xTB_Nucleophilicity`    | Morfeus `XTB.get_global_descriptor("nucleophilicity")`  |
| `xTB_Electrofugality`    | Morfeus `XTB.get_global_descriptor("electrofugality")`  |
| `xTB_Nucleofugality`     | Morfeus `XTB.get_global_descriptor("nucleofugality")`   |
| `xTB_Chemical_Potential` | computed from `-(IP + EA) / 2`                          |
| `xTB_Hardness`           | computed from `(IP - EA) / 2`                           |
| `xTB_Softness`           | computed from `1 / (2 * hardness)`                      |
| `xTB_Dipole`             | Morfeus `XTB.get_dipole()`, then vector norm            |
| `xTB_Charge_Calpha`      | Morfeus `XTB.get_charges()`; CN reaction center         |
| `xTB_Charge_N`           | Morfeus `XTB.get_charges()`; NN reaction center         |
| `xTB_Charge_Br`          | Morfeus `XTB.get_charges()`; CN Br dummy/leaving atom   |
| `xTB_Fukui_plus_Calpha`  | Morfeus `XTB.get_fukui("electrophilicity")`; CN center  |
| `xTB_Fukui_plus_N`       | Morfeus `XTB.get_fukui("electrophilicity")`; NN center  |
| `xTB_Fukui_minus_Calpha` | Morfeus `XTB.get_fukui("nucleophilicity")`; CN center   |
| `xTB_Fukui_minus_N`      | Morfeus `XTB.get_fukui("nucleophilicity")`; NN center   |
| `xTB_Fukui_dual_Calpha`  | Morfeus `XTB.get_fukui("dual")`; CN center              |
| `xTB_Fukui_dual_N`       | Morfeus `XTB.get_fukui("dual")`; NN center              |


In [ ]:
# Extended xTB descriptors via morfeus.XTB

def morfeus_xtb_descriptors(xyz_file, mol_type=None):
    elements, coords = _read_morfeus_geometry(xyz_file)
    xtb = XTB(elements, coords)
    desc = {}
    try:
        homo = xtb.get_homo()
        lumo = xtb.get_lumo()
        desc['xTB_HOMO'] = float(homo)
        desc['xTB_LUMO'] = float(lumo)
        desc['xTB_HOMO_LUMO_gap'] = float(lumo - homo)
    except Exception:
        pass
    try:
        desc['xTB_IP'] = float(xtb.get_ip(corrected=True))
        desc['xTB_EA'] = float(xtb.get_ea(corrected=True))
    except Exception:
        pass
    for variety, col in [
        ('electrophilicity', 'xTB_Electrophilicity'),
        ('nucleophilicity', 'xTB_Nucleophilicity'),
        ('electrofugality', 'xTB_Electrofugality'),
        ('nucleofugality', 'xTB_Nucleofugality'),
    ]:
        try:
            val = xtb.get_global_descriptor(variety)
            desc[col] = float(val)
        except Exception:
            pass
    try:
        ip = desc.get('xTB_IP')
        ea = desc.get('xTB_EA')
        if ip is not None and ea is not None:
            desc['xTB_Chemical_Potential'] = float(-(ip + ea) / 2.0)
            desc['xTB_Hardness'] = float((ip - ea) / 2.0)
            eta = desc.get('xTB_Hardness')
            if eta and eta > 0:
                desc['xTB_Softness'] = float(1.0 / (2.0 * eta))
    except Exception:
        pass
    try:
        dip = xtb.get_dipole()
        desc['xTB_Dipole'] = float(np.linalg.norm(dip)) if hasattr(dip, '__len__') else float(dip)
    except Exception:
        pass
    # Local charges and Fukui values at ligand reaction center
    if mol_type is not None:
        centers = find_reaction_center(elements, coords, mol_type)
        center_idx = centers.get('center_idx')
        dummy_idx = centers.get('dummy_idx')
    else:
        center_idx = None
        dummy_idx = None
    if center_idx:
        try:
            charges = xtb.get_charges()
            # morfeus/xtb wrappers may be 1-indexed
            try:
                desc[f'xTB_Charge_{"Calpha" if mol_type=="CN" else "N"}'] = float(charges[center_idx])
            except Exception:
                try:
                    desc[f'xTB_Charge_{"Calpha" if mol_type=="CN" else "N"}'] = float(charges[center_idx-1])
                except Exception:
                    pass
            if dummy_idx and mol_type == 'CN':
                try:
                    desc['xTB_Charge_Br'] = float(charges[dummy_idx])
                except Exception:
                    try:
                        desc['xTB_Charge_Br'] = float(charges[dummy_idx-1])
                    except Exception:
                        pass
        except Exception:
            pass
        for variety, col_suffix in [
            ('electrophilicity', f'xTB_Fukui_plus_{"Calpha" if mol_type=="CN" else "N"}'),
            ('nucleophilicity',  f'xTB_Fukui_minus_{"Calpha" if mol_type=="CN" else "N"}'),
            ('dual',              f'xTB_Fukui_dual_{"Calpha" if mol_type=="CN" else "N"}'),
        ]:
            try:
                f = xtb.get_fukui(variety)
                try:
                    desc[col_suffix] = float(f[center_idx])
                except Exception:
                    try:
                        desc[col_suffix] = float(f[center_idx-1])
                    except Exception:
                        pass
            except Exception:
                pass
    return desc

print('Appended Morfeus/xTB advanced feature functions')

In [ ]:
# Morfeus and xTB wrappers (unified runner)
def compute_morfeus_descriptors(xyz_file, mol_type='CN', run_xtb=True):
    """
    Run all advanced Morfeus/XTB feature functions and return a merged dict.

    This wrapper is for free CN/NN ligand XYZ files.

    It calls:
        - morfeus_dispersion
        - morfeus_sasa
        - morfeus_buried_volume
        - morfeus_sterimol
        - morfeus_xtb_descriptors, optional

    It does NOT call:
        - morfeus_solid_and_cone
        - morfeus_bite_angle

    because those require a real Ir complex / metal center.
    """

    if xyz_file is None:
        return {}

    results = {}

    modules = [
        ("morfeus_dispersion", morfeus_dispersion, {}),
        ("morfeus_sasa", morfeus_sasa, {}),
        ("morfeus_buried_volume", morfeus_buried_volume, {"mol_type": mol_type}),
        ("morfeus_sterimol", morfeus_sterimol, {"mol_type": mol_type})
        #("morfeus_solid_cone", morfeus_solid_and_cone, {"mol_type": mol_type}),
        #("morfeus_bite", morfeus_bite_angle, {"mol_type": mol_type}),
    ]

    if run_xtb:
        modules.append(
            ("morfeus_xtb", morfeus_xtb_descriptors, {"mol_type": mol_type})
        )

    for name, func, kwargs in modules:
        try:
            results.update(func(xyz_file, **kwargs) or {})
        except Exception as e:
            results[f"{name}_error"] = str(e)

    return results

print("Unified ligand-only Morfeus runner defined")

Unified Morfeus runner defined


## Processing Function

In [ ]:
# Per-ligand processing wrapper
def process_ligand(ligand_id, mol_type, conformers_dir, write_xyz_to=None):
    if write_xyz_to is None:
        write_xyz_to = Path('output') / 'tmp_smoke_xyz'
    sdf_name = f"{ligand_id}_conformers.sdf"
    sdf_path = Path(conformers_dir) / sdf_name
    result = {'ligand_id': ligand_id, 'mol_type': mol_type, 'sdf_file': str(sdf_path)}
    if not sdf_path.exists():
        result['status'] = 'missing_sdf'
        return result
    mol, energy = read_sdf_pick_lowest_energy(sdf_path)
    if mol is None:
        result['status'] = 'no_mol'
        return result
# try to sanitise / get smiles
    try:
        smiles = Chem.MolToSmiles(Chem.Mol(mol))
    except Exception:
        try:
            smiles = Chem.MolToSmiles(Chem.RemoveHs(mol))
        except Exception:
            smiles = None
    result['smiles'] = smiles
# write xyz for downstream tools
    xyz_path = Path(write_xyz_to) / f"{ligand_id}_conformer.xyz"
    try:
        xyz_written = write_xyz(mol, xyz_path, energy)
        result['xyz_file'] = xyz_written
    except Exception as e:
        result['xyz_error'] = str(e)
# rdkit descriptors
    try:
        rd_desc = compute_rdkit_descriptors(mol)
        result.update(rd_desc)
    except Exception as e:
        result['rdkit_error'] = str(e)
# Mordred descriptors
    try:
        mordred_desc = compute_mordred_descriptors(mol)
        result.update(mordred_desc)
    except Exception as e:
        result["mordred_error"] = str(e)
# Morfeus / xTB (best-effort) - unified runner
    try:
        morf = compute_morfeus_descriptors(result.get('xyz_file'), mol_type, run_xtb=True)
        result.update(morf)
    except Exception as e:
        result['morfeus_error'] = str(e)
    result['status'] = 'ok'
    return result

print('process_ligand defined')

def read_ligand_list(csv_path, prefix, fallback_dir=None, sample_n=None):
    """Read ligand identifiers from CSV or fallback conformer directory."""
    ids = []
    raw_map = {}
    csv_path = Path(csv_path)

    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path, dtype=str, comment="#")
        except Exception:
            df = pd.read_csv(csv_path, dtype=str, engine="python")

        candidate_cols = [c for c in df.columns if re.search("id|name|ligand", c, re.I)]
        if candidate_cols:
            id_col = candidate_cols[0]
        else:
            candidate_cols = [c for c in df.columns if re.search("smile", c, re.I)]
            id_col = candidate_cols[0] if candidate_cols else df.columns[0]

        for i, raw in enumerate(df[id_col].fillna("").astype(str).tolist()):
            raw_s = raw.strip()
            m = re.search(r"(\d+)(?!.*\d)", raw_s)
            idx = m.group(1) if m else str(i + 1)
            ligand_id = f"{prefix}_{idx}"

            # Ensure uniqueness if duplicate IDs appear in the CSV
            if ligand_id in ids:
                ligand_id = f"{prefix}_{idx}_{len(ids) + 1}"

            ids.append(ligand_id)
            raw_map[ligand_id] = raw_s

    else:
        if fallback_dir and Path(fallback_dir).exists():
            for p in sorted(Path(fallback_dir).glob(f"{prefix}_*conformers.sdf")):
                m = re.search(rf"{prefix}_(\d+)", p.name)
                ligand_id = f"{prefix}_{m.group(1)}" if m else p.stem.replace("_conformers", "")
                if ligand_id in ids:
                    continue
                ids.append(ligand_id)
                raw_map[ligand_id] = p.name

    if sample_n is not None:
        ids = ids[:sample_n]

    return ids, raw_map

def _safe_process_and_strip(res):
    """Remove non-feature metadata and make list/dict values CSV-friendly."""
    if not res:
        return {}

    # Keep smiles and mol_type as metadata, but remove file/status fields.
    bad_keys = {
        "status",
        "sdf_file",
        "ligand_id",
        "xyz_file",
    }

    out = {}
    for k, v in res.items():
        if k in bad_keys:
            continue

        if isinstance(v, (list, dict)):
            try:
                out[k] = json.dumps(v)
            except Exception:
                out[k] = str(v)
        else:
            out[k] = v

    return out


process_ligand defined


In [ ]:
def compute_ligand_feature_dict(ligand_ids, mol_type, conformer_dir, tmp_xyz_dir):
    """Compute descriptor dictionary once per ligand."""
    features = {}

    for ligand_id in ligand_ids:
        try:
            print(f"Processing {mol_type}:", ligand_id)

            res = process_ligand(
                ligand_id,
                mol_type,
                conformer_dir,
                write_xyz_to=tmp_xyz_dir,
            )

            features[ligand_id] = _safe_process_and_strip(res)

        except Exception as e:
            features[ligand_id] = {"error": str(e)}

    return features


def build_pairwise_descriptor_table(cn_ids, nn_ids, cn_features, nn_features):
    """Build CN-NN pairwise descriptor table."""
    rows = []

    for cid in cn_ids:
        for nid in nn_ids:
            row = {
                "combo": f"{cid}#{nid}",
                "CN_ligand_id": cid,
                "NN_ligand_id": nid,
            }

            for k, v in cn_features.get(cid, {}).items():
                row[f"CN_{k}"] = v

            for k, v in nn_features.get(nid, {}).items():
                row[f"NN_{k}"] = v

            rows.append(row)

    return pd.DataFrame(rows)


def merge_activity_score(descriptor_df, target_csv, output_file=None):
    """Merge activity_score into descriptor table if target CSV exists."""
    if not Path(target_csv).exists():
        print("No target CSV found. Returning descriptor-only table.")
        return descriptor_df.copy()

    target_df = pd.read_csv(target_csv)

    if "combo" not in target_df.columns:
        raise ValueError("Target CSV must contain a 'combo' column.")

    possible_target_cols = [
        "activity_score",
        "reduction_activity_score_1_fast_0_slow",
    ]

    target_cols = [c for c in possible_target_cols if c in target_df.columns]

    if not target_cols:
        raise ValueError(f"Target CSV must contain one of: {possible_target_cols}")

    target_col = target_cols[0]

    out_df = descriptor_df.merge(
        target_df[["combo", target_col]],
        on="combo",
        how="left",
    )

    if target_col != "activity_score":
        out_df = out_df.rename(columns={target_col: "activity_score"})

    missing_targets = out_df["activity_score"].isna().sum()
    print("Missing activity_score values after merge:", missing_targets)

    if output_file is not None:
        out_df.to_csv(output_file, index=False)
        print("Wrote descriptors + activity_score to:")
        print(output_file)

    return out_df



## Main